In [1]:
import pandas as pd 

In [4]:
# Load HPI full file
hpi = pd.read_csv("Data/UK-HPI-full-file-2025-10 (1).csv")

# Keep only UK-wide rows
hpi_uk = hpi[hpi["RegionName"] == "United Kingdom"].copy()

# Keep only Date and Index (Index = HPI)
hpi_uk = hpi_uk[["Date", "Index"]].rename(columns={"Index": "HPI"})

# Convert Date to monthly format YYYY-MM
hpi_uk["Date"] = pd.to_datetime(hpi_uk["Date"], dayfirst=True, errors="coerce")
hpi_uk = hpi_uk.dropna(subset=["Date"])
hpi_uk["Date"] = hpi_uk["Date"].dt.to_period("M").astype(str)

# Sort by date
hpi_uk = hpi_uk.sort_values("Date").reset_index(drop=True)

# Check result
hpi_uk


,Date,HPI
0,1968-04,1.3
1,1968-05,1.3
2,1968-06,1.3
3,1968-07,1.3
4,1968-08,1.3
...,...,...
686,2025-06,102.9
687,2025-07,103.6
688,2025-08,104.3
689,2025-09,103.5


In [3]:
# Load base rate data
base_rate = pd.read_csv("Data/interestRates.csv")

# Clean column names (removes hidden spaces)
base_rate.columns = base_rate.columns.str.strip()

# Check columns (optional, for sanity)
base_rate.columns

base_rate = base_rate.rename(columns={
    base_rate.columns[0]: "Date",
    base_rate.columns[1]: "BaseRate"
})

base_rate["Date"] = pd.to_datetime(base_rate["Date"], dayfirst=True, errors="coerce")

# Drop any bad rows
base_rate = base_rate.dropna(subset=["Date"])

base_rate = base_rate.sort_values("Date")

base_rate_monthly = (
    base_rate
    .set_index("Date")[["BaseRate"]]
    .resample("MS")   # Monthly (month start)
    .ffill()
    .reset_index()
)

# Convert Date to YYYY-MM string
base_rate_monthly["Date"] = base_rate_monthly["Date"].dt.to_period("M").astype(str)

base_rate_monthly

C:\Users\izaan\AppData\Local\Temp\ipykernel_5728\1333903661.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  base_rate["Date"] = pd.to_datetime(base_rate["Date"], dayfirst=True, errors="coerce")


,Date,BaseRate
0,1976-01,NaN
1,1976-02,10.50
2,1976-03,9.25
3,1976-04,9.00
4,1976-05,10.50
...,...,...
1195,2075-08,11.00
1196,2075-09,11.00
1197,2075-10,11.00
1198,2075-11,12.00


In [5]:
import pandas as pd

# Load AWE data
awe_raw = pd.read_csv("Data/emp.csv")

# Clean column names
awe_raw.columns = awe_raw.columns.str.strip()

# Pick the AWE column
AWE_COL = "AWE: Whole Economy Real Terms Level (£): Seasonally Adjusted Total Pay"

# Keep only Date + AWE
awe = awe_raw[["Title", AWE_COL]].copy()

# Rename columns
awe = awe.rename(columns={
    "Title": "Date",
    AWE_COL: "AWE"
})

# 🔽 DROP FIRST 33 ROWS
awe = awe.iloc[31:].reset_index(drop=True)

awe


,Date,AWE
0,2000 Q1,419
1,2000 Q2,425
2,2000 Q3,429
3,2000 Q4,434
4,2001 Q1,440
...,...,...
853,2025 JUL,527
854,2025 AUG,529
855,2025 SEP,528
856,2025 OCT,532


In [6]:
master = hpi_uk.merge(base_rate_monthly, on="Date", how="inner")

master

,Date,HPI,BaseRate
0,1976-01,3.5,NaN
1,1976-02,3.5,10.50
2,1976-03,3.5,9.25
3,1976-04,3.6,9.00
4,1976-05,3.6,10.50
...,...,...,...
593,2025-06,102.9,4.25
594,2025-07,103.6,4.25
595,2025-08,104.3,4.25
596,2025-09,103.5,4.00


In [7]:
awe = awe.iloc[103:]

In [8]:
awe["Date"] = awe["Date"].astype(str).str.strip().str.upper()

C:\Users\izaan\AppData\Local\Temp\ipykernel_5728\1285658983.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  awe["Date"] = awe["Date"].astype(str).str.strip().str.upper()


In [9]:
awe

,Date,AWE
103,1963 JAN,NaN
104,1963 FEB,NaN
105,1963 MAR,NaN
106,1963 APR,NaN
107,1963 MAY,NaN
...,...,...
853,2025 JUL,527
854,2025 AUG,529
855,2025 SEP,528
856,2025 OCT,532


In [ ]:
awe["Date"] = pd.to_datetime(awe["Date"], format="%Y %b", errors="coerce")

awe

C:\Users\izaan\AppData\Local\Temp\ipykernel_24236\598483056.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  awe["Date"] = pd.to_datetime(awe["Date"], format="%Y %b", errors="coerce")


,Date,AWE
103,1963-01-01,NaN
104,1963-02-01,NaN
105,1963-03-01,NaN
106,1963-04-01,NaN
107,1963-05-01,NaN
...,...,...
853,2025-07-01,527
854,2025-08-01,529
855,2025-09-01,528
856,2025-10-01,532


In [10]:
awe


,Date,AWE
103,1963 JAN,NaN
104,1963 FEB,NaN
105,1963 MAR,NaN
106,1963 APR,NaN
107,1963 MAY,NaN
...,...,...
853,2025 JUL,527
854,2025 AUG,529
855,2025 SEP,528
856,2025 OCT,532


In [11]:
hpi_uk = hpi_uk.drop_duplicates(subset=["Date"], keep="last")
base_rate_monthly = base_rate_monthly.drop_duplicates(subset=["Date"], keep="last")
awe = awe.drop_duplicates(subset=["Date"], keep="last")

In [12]:
master

,Date,HPI,BaseRate
0,1976-01,3.5,NaN
1,1976-02,3.5,10.50
2,1976-03,3.5,9.25
3,1976-04,3.6,9.00
4,1976-05,3.6,10.50
...,...,...,...
593,2025-06,102.9,4.25
594,2025-07,103.6,4.25
595,2025-08,104.3,4.25
596,2025-09,103.5,4.00


In [13]:
#SummaryVersion1.csv
summary = pd.read_csv("SummaryVersion1.csv")
summary


,Date,HPI,BaseRate,AWE
0,1976-01,3.5,NaN,NaN
1,1976-02,3.5,10.50,NaN
2,1976-03,3.5,9.25,NaN
3,1976-04,3.6,9.00,NaN
4,1976-05,3.6,10.50,NaN
...,...,...,...,...
593,2025-06,102.9,4.25,527.0
594,2025-07,103.6,4.25,527.0
595,2025-08,104.3,4.25,529.0
596,2025-09,103.5,4.00,528.0


In [14]:
summary["BaseRate_lag6"] = summary["BaseRate"].shift(6)
summary["BaseRate_lag12"] = summary["BaseRate"].shift(12)

In [15]:
summary

,Date,HPI,BaseRate,AWE,BaseRate_lag6,BaseRate_lag12
0,1976-01,3.5,NaN,NaN,NaN,NaN
1,1976-02,3.5,10.50,NaN,NaN,NaN
2,1976-03,3.5,9.25,NaN,NaN,NaN
3,1976-04,3.6,9.00,NaN,NaN,NaN
4,1976-05,3.6,10.50,NaN,NaN,NaN
...,...,...,...,...,...,...
593,2025-06,102.9,4.25,527.0,4.75,5.25
594,2025-07,103.6,4.25,527.0,4.75,5.25
595,2025-08,104.3,4.25,529.0,4.75,5.00
596,2025-09,103.5,4.00,528.0,4.50,5.00


In [16]:
summary = summary.dropna().copy()

In [17]:
summary

,Date,HPI,BaseRate,AWE,BaseRate_lag6,BaseRate_lag12
288,2000-01,29.9,5.50,420.0,5.00,6.25
289,2000-02,29.9,5.75,411.0,5.00,6.00
290,2000-03,29.9,6.00,425.0,5.00,5.50
291,2000-04,31.5,6.00,423.0,5.25,5.50
292,2000-05,31.5,6.00,426.0,5.25,5.25
...,...,...,...,...,...,...
593,2025-06,102.9,4.25,527.0,4.75,5.25
594,2025-07,103.6,4.25,527.0,4.75,5.25
595,2025-08,104.3,4.25,529.0,4.75,5.00
596,2025-09,103.5,4.00,528.0,4.50,5.00


In [18]:
summary[["HPI", "BaseRate_lag6", "BaseRate_lag12"]].corr()

summary.to_csv("SummaryVersion2.csv", index=False)

In [19]:
summary

,Date,HPI,BaseRate,AWE,BaseRate_lag6,BaseRate_lag12
288,2000-01,29.9,5.50,420.0,5.00,6.25
289,2000-02,29.9,5.75,411.0,5.00,6.00
290,2000-03,29.9,6.00,425.0,5.00,5.50
291,2000-04,31.5,6.00,423.0,5.25,5.50
292,2000-05,31.5,6.00,426.0,5.25,5.25
...,...,...,...,...,...,...
593,2025-06,102.9,4.25,527.0,4.75,5.25
594,2025-07,103.6,4.25,527.0,4.75,5.25
595,2025-08,104.3,4.25,529.0,4.75,5.00
596,2025-09,103.5,4.00,528.0,4.50,5.00


In [53]:
housingSupply = pd.read_excel("Data/LiveTable222.ods", engine="odf")

housingSupply.columns = housingSupply.iloc[0]
housingSupply = housingSupply.drop(0)

housingSupply = housingSupply[["Period" , "All Starts" , "All Completions"]]

housingSupply

,Period,All Starts,All Completions
1,2000 Q1,37210,34930
2,2000 Q2,36510,34860
3,2000 Q3,34920,33070
4,2000 Q4,33520,32420
5,2001 Q1,35280,33060
...,...,...,...
99,2024 Q3,28850,36180
100,2024 Q4,30420,36920
101,2025 Q1,28570,35770
102,2025 Q2,29650,35380


In [54]:
summary

,Date,HPI,BaseRate,AWE,BaseRate_lag6,BaseRate_lag12
288,2000-01,29.9,5.50,420.0,5.00,6.25
289,2000-02,29.9,5.75,411.0,5.00,6.00
290,2000-03,29.9,6.00,425.0,5.00,5.50
291,2000-04,31.5,6.00,423.0,5.25,5.50
292,2000-05,31.5,6.00,426.0,5.25,5.25
...,...,...,...,...,...,...
593,2025-06,102.9,4.25,527.0,4.75,5.25
594,2025-07,103.6,4.25,527.0,4.75,5.25
595,2025-08,104.3,4.25,529.0,4.75,5.00
596,2025-09,103.5,4.00,528.0,4.50,5.00


In [55]:
import pandas as pd

# Split Period into year and quarter
year = housingSupply["Period"].str.split().str[0]
quarter = housingSupply["Period"].str.split().str[1]

# Map quarter to month
quarter_to_month = {
    "Q1": "01",
    "Q2": "04",
    "Q3": "07",
    "Q4": "10"
}

month = quarter.map(quarter_to_month)

# Create YYYY-MM format
housingSupply["Period"] = year + "-" + month

# (Optional but recommended) convert to datetime
housingSupply["Period"] = pd.to_datetime(housingSupply["Period"], format="%Y-%m")

In [57]:
housingSupply["Period"] = housingSupply["Period"].dt.to_period("M").astype(str)

housingSupply

,Period,All Starts,All Completions
1,2000-01,37210,34930
2,2000-04,36510,34860
3,2000-07,34920,33070
4,2000-10,33520,32420
5,2001-01,35280,33060
...,...,...,...
99,2024-07,28850,36180
100,2024-10,30420,36920
101,2025-01,28570,35770
102,2025-04,29650,35380


In [58]:
summary

,Date,HPI,BaseRate,AWE,BaseRate_lag6,BaseRate_lag12
288,2000-01,29.9,5.50,420.0,5.00,6.25
289,2000-02,29.9,5.75,411.0,5.00,6.00
290,2000-03,29.9,6.00,425.0,5.00,5.50
291,2000-04,31.5,6.00,423.0,5.25,5.50
292,2000-05,31.5,6.00,426.0,5.25,5.25
...,...,...,...,...,...,...
593,2025-06,102.9,4.25,527.0,4.75,5.25
594,2025-07,103.6,4.25,527.0,4.75,5.25
595,2025-08,104.3,4.25,529.0,4.75,5.00
596,2025-09,103.5,4.00,528.0,4.50,5.00


In [63]:
housingSupply = housingSupply.rename(columns={
    "Period": "Date"
})

summary["Date"] = summary["Date"].astype(str)
housingSupply["Date"] = housingSupply["Date"].astype(str)

summary = summary.merge(
    housingSupply[["Date", "All Starts", "All Completions"]],
    on="Date",
    how="left"
)

summary

,Date,HPI,BaseRate,AWE,BaseRate_lag6,BaseRate_lag12,All Starts_x,All Completions_x,All Starts_y,All Completions_y
0,2000-01,29.9,5.50,420.0,5.00,6.25,37210,34930,37210,34930
1,2000-02,29.9,5.75,411.0,5.00,6.00,NaN,NaN,NaN,NaN
2,2000-03,29.9,6.00,425.0,5.00,5.50,NaN,NaN,NaN,NaN
3,2000-04,31.5,6.00,423.0,5.25,5.50,36510,34860,36510,34860
4,2000-05,31.5,6.00,426.0,5.25,5.25,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
305,2025-06,102.9,4.25,527.0,4.75,5.25,NaN,NaN,NaN,NaN
306,2025-07,103.6,4.25,527.0,4.75,5.25,29620,33020,29620,33020
307,2025-08,104.3,4.25,529.0,4.75,5.00,NaN,NaN,NaN,NaN
308,2025-09,103.5,4.00,528.0,4.50,5.00,NaN,NaN,NaN,NaN


In [64]:
# drop the duplicate pair
summary = summary.drop(columns=["All Starts_y", "All Completions_y"])

# rename the kept pair
summary = summary.rename(columns={
    "All Starts_x": "All Starts",
    "All Completions_x": "All Completions"
})

In [65]:
summary

,Date,HPI,BaseRate,AWE,BaseRate_lag6,BaseRate_lag12,All Starts,All Completions
0,2000-01,29.9,5.50,420.0,5.00,6.25,37210,34930
1,2000-02,29.9,5.75,411.0,5.00,6.00,NaN,NaN
2,2000-03,29.9,6.00,425.0,5.00,5.50,NaN,NaN
3,2000-04,31.5,6.00,423.0,5.25,5.50,36510,34860
4,2000-05,31.5,6.00,426.0,5.25,5.25,NaN,NaN
...,...,...,...,...,...,...,...,...
305,2025-06,102.9,4.25,527.0,4.75,5.25,NaN,NaN
306,2025-07,103.6,4.25,527.0,4.75,5.25,29620,33020
307,2025-08,104.3,4.25,529.0,4.75,5.00,NaN,NaN
308,2025-09,103.5,4.00,528.0,4.50,5.00,NaN,NaN


In [67]:
# make sure dates are in order
summary = summary.sort_values("Date")

# forward fill the quarterly housing columns
summary[["All Starts", "All Completions"]] = (
    summary[["All Starts", "All Completions"]].ffill()
)

summary

,Date,HPI,BaseRate,AWE,BaseRate_lag6,BaseRate_lag12,All Starts,All Completions
0,2000-01,29.9,5.50,420.0,5.00,6.25,37210,34930
1,2000-02,29.9,5.75,411.0,5.00,6.00,37210,34930
2,2000-03,29.9,6.00,425.0,5.00,5.50,37210,34930
3,2000-04,31.5,6.00,423.0,5.25,5.50,36510,34860
4,2000-05,31.5,6.00,426.0,5.25,5.25,36510,34860
...,...,...,...,...,...,...,...,...
305,2025-06,102.9,4.25,527.0,4.75,5.25,29650,35380
306,2025-07,103.6,4.25,527.0,4.75,5.25,29620,33020
307,2025-08,104.3,4.25,529.0,4.75,5.00,29620,33020
308,2025-09,103.5,4.00,528.0,4.50,5.00,29620,33020


In [68]:
years = list(range(2015, 2025))
values = ["25,309","191,952","74,884","65,860","42,234","34,777","39,050","36,772","41,332","13,370"]

yearly = pd.DataFrame({
    "Year": years,
    "Investment_million": pd.to_numeric([v.replace(",", "") for v in values], errors="coerce")
})

In [69]:
summary["Year"] = summary["Date"].str.slice(0, 4).astype(int)

In [70]:
summary = summary.merge(yearly, on="Year", how="left")

In [71]:
summary = summary.drop(columns=["Year"])

In [72]:
summary

,Date,HPI,BaseRate,AWE,BaseRate_lag6,BaseRate_lag12,All Starts,All Completions,Investment_million
0,2000-01,29.9,5.50,420.0,5.00,6.25,37210,34930,NaN
1,2000-02,29.9,5.75,411.0,5.00,6.00,37210,34930,NaN
2,2000-03,29.9,6.00,425.0,5.00,5.50,37210,34930,NaN
3,2000-04,31.5,6.00,423.0,5.25,5.50,36510,34860,NaN
4,2000-05,31.5,6.00,426.0,5.25,5.25,36510,34860,NaN
...,...,...,...,...,...,...,...,...,...
305,2025-06,102.9,4.25,527.0,4.75,5.25,29650,35380,NaN
306,2025-07,103.6,4.25,527.0,4.75,5.25,29620,33020,NaN
307,2025-08,104.3,4.25,529.0,4.75,5.00,29620,33020,NaN
308,2025-09,103.5,4.00,528.0,4.50,5.00,29620,33020,NaN


In [73]:
summary.tail(12)

,Date,HPI,BaseRate,AWE,BaseRate_lag6,BaseRate_lag12,All Starts,All Completions,Investment_million
298,2024-11,101.4,5.00,525.0,5.25,5.25,30420,36920,13370.0
299,2024-12,101.2,4.75,529.0,5.25,5.25,30420,36920,13370.0
300,2025-01,101.7,4.75,522.0,5.25,5.25,28570,35770,NaN
301,2025-02,101.9,4.75,528.0,5.00,5.25,28570,35770,NaN
302,2025-03,103.3,4.50,534.0,5.00,5.25,28570,35770,NaN
303,2025-04,100.5,4.50,527.0,5.00,5.25,29650,35380,NaN
304,2025-05,101.7,4.50,527.0,5.00,5.25,29650,35380,NaN
305,2025-06,102.9,4.25,527.0,4.75,5.25,29650,35380,NaN
306,2025-07,103.6,4.25,527.0,4.75,5.25,29620,33020,NaN
307,2025-08,104.3,4.25,529.0,4.75,5.00,29620,33020,NaN


In [75]:
CPI = pd.read_excel("Data/CPIH_monthly_2000.xlsx")

CPI

,Month,CPIH_Index
0,2000-01,69.0
1,2000-01,68.4
2,2000-02,68.5
3,2000-03,68.5
4,2000-04,68.8
...,...,...
333,2025-08,140.6
334,2025-09,141.0
335,2025-10,141.7
336,2025-11,142.3


In [76]:
CPI = CPI.rename(columns={"Month": "Date"})

In [77]:
summary["Date"] = summary["Date"].astype(str)
CPI["Date"] = CPI["Date"].astype(str)

In [78]:
summary = summary.merge(
    CPI[["Date", "CPIH_Index"]],
    on="Date",
    how="left"
)

In [79]:
summary

,Date,HPI,BaseRate,AWE,BaseRate_lag6,BaseRate_lag12,All Starts,All Completions,Investment_million,CPIH_Index
0,2000-01,29.9,5.50,420.0,5.00,6.25,37210,34930,NaN,69.0
1,2000-01,29.9,5.50,420.0,5.00,6.25,37210,34930,NaN,68.4
2,2000-02,29.9,5.75,411.0,5.00,6.00,37210,34930,NaN,68.5
3,2000-03,29.9,6.00,425.0,5.00,5.50,37210,34930,NaN,68.5
4,2000-04,31.5,6.00,423.0,5.25,5.50,36510,34860,NaN,68.8
...,...,...,...,...,...,...,...,...,...,...
331,2025-06,102.9,4.25,527.0,4.75,5.25,29650,35380,NaN,140.8
332,2025-07,103.6,4.25,527.0,4.75,5.25,29620,33020,NaN,140.2
333,2025-08,104.3,4.25,529.0,4.75,5.00,29620,33020,NaN,140.6
334,2025-09,103.5,4.00,528.0,4.50,5.00,29620,33020,NaN,141.0
